# Knowledge Distillation

[← Back to lesson](https://ml-viz-ruby.vercel.app/courses/fine-tuning-alignment/06-knowledge-distillation)

This notebook implements knowledge distillation from scratch — soft labels, temperature scaling, and the KL divergence loss — and compares a distilled student to an identically-sized student trained from scratch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.edgecolor': '#2d3748',
    'grid.color': '#2d3748',
    'axes.grid': True,
})

np.random.seed(42)

## 1. Soft labels vs. hard labels

Suppose a teacher model outputs logits for a 6-class problem. We compare hard labels (one-hot) with soft labels at different temperatures.

In [ ]:
def softmax(logits, tau=1.0):
    logits = np.array(logits, dtype=float)
    logits = logits / tau
    logits -= logits.max()  # numerical stability
    exp = np.exp(logits)
    return exp / exp.sum()

# Teacher logits for 6 classes (true class = 0)
# Class 0: 'cat', 1: 'kitten', 2: 'feline', 3: 'dog', 4: 'car', 5: 'airplane'
teacher_logits = np.array([5.2, 3.8, 2.9, 1.1, 0.4, -0.3])
class_names = ['cat', 'kitten', 'feline', 'dog', 'car', 'airplane']

hard_label = np.array([1, 0, 0, 0, 0, 0])  # one-hot
soft_tau1  = softmax(teacher_logits, tau=1)
soft_tau4  = softmax(teacher_logits, tau=4)
soft_tau10 = softmax(teacher_logits, tau=10)

fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True)
colors = ['#6366f1', '#6366f1', '#6366f1', '#6366f1']
bar_alpha = [1.0, 0.85, 0.7, 0.55]

for ax, data, title, alpha in zip(
    axes,
    [hard_label, soft_tau1, soft_tau4, soft_tau10],
    ['Hard label (τ=∞)', 'Soft τ=1', 'Soft τ=4', 'Soft τ=10'],
    bar_alpha
):
    bars = ax.bar(class_names, data, color='#6366f1', alpha=alpha)
    ax.set_title(title, color='#e2e8f0', pad=10)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, data):
        if val > 0.02:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=8, color='#94a3b8')

axes[0].set_ylabel('Probability', color='#94a3b8')
plt.suptitle('Hard vs. Soft Labels at Different Temperatures', color='#e2e8f0', y=1.02)
plt.tight_layout()
plt.show()

print(f"Information in hard label:    {-np.sum(hard_label * np.log(hard_label + 1e-9)):.2f} nats")
print(f"Information in soft τ=1:      {-np.sum(soft_tau1 * np.log(soft_tau1 + 1e-9)):.2f} nats")
print(f"Information in soft τ=4:      {-np.sum(soft_tau4 * np.log(soft_tau4 + 1e-9)):.2f} nats")
print(f"Information in soft τ=10:     {-np.sum(soft_tau10 * np.log(soft_tau10 + 1e-9)):.2f} nats")

## 2. The distillation loss

$$\mathcal{L} = \alpha \cdot \tau^2 \cdot \text{KL}(\hat{y}_T^{(\tau)} \| \hat{y}_S^{(\tau)}) + (1-\alpha) \cdot \mathcal{L}_{\text{CE}}$$

In [ ]:
def kl_divergence(p, q, eps=1e-9):
    """KL(p || q) = sum p * log(p/q)"""
    p, q = np.array(p) + eps, np.array(q) + eps
    return np.sum(p * np.log(p / q))

def cross_entropy(probs, label_idx, eps=1e-9):
    return -np.log(probs[label_idx] + eps)

def distillation_loss(teacher_logits, student_logits, true_label, tau=4.0, alpha=0.7):
    # Soft targets at temperature τ
    p_teacher = softmax(teacher_logits, tau)
    p_student  = softmax(student_logits, tau)
    
    kd  = kl_divergence(p_teacher, p_student) * (tau ** 2)
    
    # Hard CE at τ=1
    p_student_hard = softmax(student_logits, tau=1)
    ce  = cross_entropy(p_student_hard, true_label)
    
    return alpha * kd + (1 - alpha) * ce, kd, ce

# Example: student whose logits are noisier than the teacher
student_logits = teacher_logits + np.random.randn(6) * 1.5
total, kd, ce = distillation_loss(teacher_logits, student_logits, true_label=0)

print(f"Teacher logits:  {teacher_logits}")
print(f"Student logits:  {student_logits.round(2)}")
print(f"KD loss (×τ²):   {kd:.4f}")
print(f"CE loss:         {ce:.4f}")
print(f"Total loss:      {total:.4f}  (α=0.7)")

## 3. Effect of temperature on the KD gradient

How does temperature τ change the distillation loss surface?

In [ ]:
taus = [1, 2, 4, 8, 16]
noise_levels = np.linspace(0, 3, 50)

fig, ax = plt.subplots(figsize=(9, 5))

palette = ['#6366f1', '#22d3ee', '#f59e0b', '#10b981', '#f43f5e']

for tau, color in zip(taus, palette):
    kd_losses = []
    for noise in noise_levels:
        s_logits = teacher_logits + np.random.RandomState(int(noise*100)).randn(6) * noise
        p_t = softmax(teacher_logits, tau)
        p_s = softmax(s_logits, tau)
        kd_losses.append(kl_divergence(p_t, p_s) * tau**2)
    ax.plot(noise_levels, kd_losses, label=f'τ={tau}', color=color, linewidth=2)

ax.set_xlabel('Student noise level (distance from teacher)')
ax.set_ylabel('KD loss (τ² × KL)')
ax.set_title('KD Loss vs. Student-Teacher Divergence at Different Temperatures', color='#e2e8f0')
ax.legend(title='Temperature τ', title_fontsize=9)
plt.tight_layout()
plt.show()

## 4. Sequence-level distillation (LLM style)

For LLMs, the teacher generates full text responses; the student trains on them with standard next-token prediction. We simulate this with a toy vocabulary.

In [ ]:
# Toy: vocabulary of 8 tokens, sequence of 5 tokens
VOCAB_SIZE = 8
SEQ_LEN = 5

rng = np.random.RandomState(0)

# Teacher generates a high-quality sequence (its most likely tokens)
teacher_logits_seq = rng.randn(SEQ_LEN, VOCAB_SIZE) * 2
teacher_seq = teacher_logits_seq.argmax(axis=-1)  # greedy decode

print("Teacher-generated sequence:", teacher_seq)
print("\nStudent trains on these tokens as hard labels.")
print("Per-position cross-entropy loss at random student init:")

student_logits_seq = rng.randn(SEQ_LEN, VOCAB_SIZE)
for t in range(SEQ_LEN):
    p = softmax(student_logits_seq[t])
    ce = -np.log(p[teacher_seq[t]] + 1e-9)
    print(f"  Position {t}: true token={teacher_seq[t]}, student prob={p[teacher_seq[t]]:.3f}, CE={ce:.3f}")

## ✏️ Your turn

**Exercise 1 – Temperature sweep.** Implement a function that, given a fixed teacher and student logit pair, computes the total distillation loss for temperatures τ ∈ [1, 2, 4, 8, 16] with α = 0.5. Plot total loss vs. τ. At what τ is the KD component most informative?

In [ ]:
teacher = np.array([4.0, 2.0, 1.5, 0.5, -1.0])
student = np.array([3.0, 1.0, 2.0, 0.2, -0.5])
true_label = 0

taus = [1, 2, 4, 8, 16]

# TODO(you): compute total distillation loss for each τ and plot
# losses = [distillation_loss(teacher, student, true_label, tau=t, alpha=0.5)[0] for t in taus]
# plt.plot(taus, losses, ...)

In [ ]:
# Assert cell — passes silently when correct
losses_ref = [distillation_loss(teacher, student, true_label, tau=t, alpha=0.5)[0] for t in taus]
# Your losses should be a list of 5 floats, all positive
# assert len(losses) == 5 and all(l > 0 for l in losses), "Expected 5 positive loss values"
print("Reference losses:", [f"{l:.4f}" for l in losses_ref])

<details><summary>Solution</summary>

```python
losses = [distillation_loss(teacher, student, true_label, tau=t, alpha=0.5)[0] for t in taus]
plt.figure(figsize=(7, 4))
plt.plot(taus, losses, 'o-', color='#6366f1', linewidth=2)
plt.xlabel('Temperature τ')
plt.ylabel('Total distillation loss')
plt.title('Distillation Loss vs. Temperature')
plt.show()
```

The total loss typically peaks at intermediate τ (around 4–8) because the KD term (τ²×KL) grows initially as more information is surfaced, then decreases as the distribution becomes too flat to be useful.

</details>

**Exercise 2 – α sensitivity.** For fixed τ=4, vary α from 0 to 1 in steps of 0.1 and plot total loss. What happens at α=0 and α=1?

In [ ]:
alphas = np.linspace(0, 1, 11)

# TODO(you): compute and plot total loss for each α at τ=4